# レッスン 11 - エージェント間（A2A）プロトコル


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2Aプロトコルとは何ですか？

**エージェント間（Agent-to-Agent、A2A）プロトコル** は、AIエージェントが異なるフレームワークで構築されていても、または異なるサービスによってホストされていても、通信し、
お互いを検出し、協力できるようにするオープン標準です。


主な概念：

- **検出（Discovery）** – エージェントは自らの機能を記述した<em>エージェントカード</em>を公開し、
  他のエージェント（またはオーケストレーター）がタスクのための適切な専門家を見つけやすくします。
- **メッセージパッシング（Message Passing）** – エージェントは共通プロトコルを通じて構造化されたメッセージを交換します。
  そのため、あるエージェントからのリクエストが、別のエージェントの内部実装に関わらず理解され実行されます。
- **タスクライフサイクル（Task Lifecycle）** – A2Aは、<em>提出済み</em>、<em>作業中</em>、<em>完了</em>、<em>失敗</em>といった状態を定義し、
  オーケストレーターが委任されたタスクの進行状況を完全に把握できるようにします。


このレッスンでは、3つの専門分野に特化した旅行代理店（トラベルエージェント）をワークフローに結びつけ、
各エージェントが専門知識を提供し、結果を次に渡すA2Aスタイルの協力をシミュレートします。


## 専門的な旅行代理店の作成


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ワークフローによるマルチエージェント協働

3つのエージェントを、A2Aメッセージパッシングを反映する連続したワークフローとして接続します：

1. **CurrencyExchangeAgent** がユーザーのリクエストを受け取り、通貨に関する案内を生成します。
2. **ActivityPlannerAgent** が拡張されたコンテキストを受け取り、アクティビティの推奨を追加します。
3. **TravelManagerAgent** が両方の入力を統合して最終的な旅行ブリーフを作成します。


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 本番環境における A2A の理解

本番環境で A2A プロトコルは強力なクロスサービスシナリオを実現します:

| 機能 | 説明 |
|---|---|
| <strong>フレームワーク間の相互運用</strong> | あるフレームワークで構築されたエージェントが、他の任意の A2A 対応フレームワークで構築されたエージェントにタスクを委譲でき、本当の意味で組織を超えた相互運用性を可能にします。 |
| <strong>サービス境界</strong> | エージェントは別々のマイクロサービスやクラウドリージョン、さらには異なる組織に存在していてもシームレスに連携できます。 |
| <strong>動的発見</strong> | オーケストレーターは、実行時にエージェントカードのレジストリを照会し、特定のサブタスクに最適なスペシャリストを見つけることができます。 |
| **ストリーミング & プッシュ通知** | A2A はリアルタイムの進捗更新のためのサーバー送信イベント（SSE）や、長時間実行されるタスクのプッシュ通知をサポートします。 |

上で作成したワークフローは、このパターンの簡略化されたプロセス内バージョンです。実際の
展開では、各エージェントが HTTP エンドポイントを公開し、エージェントカードを発行し、
A2A JSON-RPC プロトコルを介して通信します。


## まとめ

このレッスンで学んだこと：

1. **A2Aプロトコルとは何か** — エージェント間の発見、メッセージング、タスク管理のためのオープン標準。
   。
2. <strong>専門的なエージェントの作成方法</strong> — 通貨交換エージェント、アクティビティプランナーエージェント、
   および旅行マネージャーオーケストレーター。
3. <strong>エージェントをワークフローに組み込む方法</strong> — `WorkflowBuilder` を使って
   エージェント間の順次メッセージ伝達をモデル化する。
4. **A2Aが実運用でどう機能するか** — クロスフレームワーク、クロスサービスの連携を
   動的発見とストリーミング更新で可能にする。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
